# 🚀 Last Layer Fine-Tuning SigLIP2-SO400M on `folds_v3.csv`
Notebook ini dirancang untuk menjalankan retraining **hanya pada Last Layer (Linear Probing)** menggunakan dataset hasil pembersihan tahap 3 (`folds_v3.csv`). Eksperimen paralel ini membandingkan apakah melatih blok penuh (LoRA) benar-benar lebih superior dibanding melatih ujung classifier-nya saja.

In [ ]:
import os, sys, shutil, glob
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Clone atau Pull Repo Terbaru
REPO_DIR = '/content/satria-data-bdcugm02'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# 3. Upgrade Torchao untuk PEFT & Install Dependensi Pinned
!pip install -q -U "torchao>=0.16.0"
!pip install -q --no-warn-conflicts -r {REPO_DIR}/track_b/requirements.txt

# 4. Copy Cepat Gambar Dataset ke Storage Lokal Colab (/tmp) dengan 32 Workers
DRIVE_TRAIN_DIR = '/content/drive/MyDrive/BDC2026/train'
LOCAL_TRAIN_DIR = '/tmp/dataset/train'

def copy_img_worker(args):
    src, dst = args
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
        shutil.copy2(src, dst)

if os.path.exists(DRIVE_TRAIN_DIR):
    print("🚀 Memulai copy cepat SELURUH GAMBAR ke storage lokal Colab (/tmp) dengan 32 workers...")
    all_imgs = []
    for root, _, files in os.walk(DRIVE_TRAIN_DIR):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                all_imgs.append(os.path.join(root, f))
                
    files_to_copy = [(src, os.path.join(LOCAL_TRAIN_DIR, os.path.relpath(src, DRIVE_TRAIN_DIR))) for src in all_imgs]
    
    with ThreadPoolExecutor(max_workers=32) as executor:
        list(executor.map(copy_img_worker, files_to_copy))
    print(f"✅ Selesai meng-copy {len(files_to_copy)} gambar ke storage lokal Colab: {LOCAL_TRAIN_DIR}")
else:
    print("⚠️ Folder gambar Drive belum ditemukan di path standar.")


## Training Last Layer dengan Data `v3`
Perbedaan utama ada di `VARIANT = 'last_layer'`.

In [ ]:
import sys, importlib
sys.path.insert(0, '/content/satria-data-bdcugm02/track_a/src')
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/src')
sys.path.insert(0, '/content/satria-data-bdcugm02/track_b/experiments')

import embed, lora_ft, config
importlib.reload(config)
importlib.reload(embed)
importlib.reload(lora_ft)

from config import CFG, make_cfg

VARIANT = 'last_layer'  # <-- Ini yang membedakan dengan LoRA
CHECKPOINT = 'google/siglip2-so400m-patch14-384'

# Gunakan folds_v3_csv untuk run ini
cfg = make_cfg(
    run_name=f'{VARIANT}_ft_fold0_5ep_v3',
    folds_csv=CFG.folds_v3_csv,   # <-- Train & evaluasi hanya di v3
    batch=8,
    accum_steps=4
)

# Menjalankan 5 Epoch Last Layer Fine-Tuning
result = lora_ft.run_smoke_test_fold0(
    variant=VARIANT,
    cfg=cfg,
    checkpoint=CHECKPOINT,
    max_epochs=5,
    n_last_blocks=4
)
result
